In [1]:
# Importing the Libraries
import pandas as pd
import numpy as np

In [3]:
# Loading the Data
data = pd.read_csv('Amazon Recommendation System.csv')

In [5]:
# Analyse the Top 5 rows of the Data
data.head()

,AKM1MP6P0OYPR,0132793040,5.0,1365811200
0,A2CX7LUOHB2NDG,0321732944,5.0,1341100800
1,A2NWSAGRHCP8N5,0439886341,1.0,1367193600
2,A2WNBOD3WNDNKT,0439886341,3.0,1374451200
3,A1GI0U4ZRJA8WN,0439886341,1.0,1334707200
4,A1QGNMC6O1VW39,0511189877,5.0,1397433600


In [7]:
# The dataset that I am using here does not have columns names, so let’s give the most appropriate names to these columns
data.columns = ['user_id', 'product_id','ratings','timestamp']

In [9]:
# This dataset is very large so let's select a sample
df = data[:int(len(data) * .1)]

In [11]:
# Now let’s prepare the dataset for creating a recommendation system

counts = df['user_id'].value_counts()
data = df[df['user_id'].isin(counts[counts >= 50].index)]
data.groupby('product_id')['ratings'].mean().sort_values(ascending=False) 
final_ratings = data.pivot(index = 'user_id', columns ='product_id', values = 'ratings').fillna(0)

num_of_ratings = np.count_nonzero(final_ratings)
possible_ratings = final_ratings.shape[0] * final_ratings.shape[1]
density = (num_of_ratings/possible_ratings)
density *= 100
final_ratings_T = final_ratings.transpose()

grouped = data.groupby('product_id').agg({'user_id': 'count'}).reset_index()
grouped.rename(columns = {'user_id': 'score'},inplace=True)
training_data = grouped.sort_values(['score', 'product_id'], ascending = [0,1]) 
training_data['Rank'] = training_data['score'].rank(ascending=0, method='first') 
recommendations = training_data.head()

In [21]:
# Now let's write a Python function to generate recommendations based on the score of the product reviews
def recommend(user_id):     
    # Make a copy to avoid modifying the original DataFrame
    recommend_products = recommendations.copy()
    
    # Add or update a user_id column
    recommend_products['user_id'] = user_id
    
    # Reorder columns to place 'user_id' first
    column_order = ['user_id'] + [col for col in recommend_products.columns if col != 'user_id']
    recommend_products = recommend_products[column_order]
    
    return recommend_products

# Test the function
print(recommend(11))


      user_id  product_id  score  Rank
113        11  B00004SB92      6   1.0
1099       11  B00008OE6I      5   2.0
368        11  B00005AW1H      4   3.0
612        11  B0000645C9      4   4.0
976        11  B00007KDVI      4   5.0


#### Summary
This is how we can create an Amazon Recommender System using Python. This dataset does not have names of products in it, it only had product id so the score of the product reviews becomes the most important feature for such kinds of datasets.